In [1]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
!pip install -q seqeval
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [2]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [3]:
!pip install -q transformers datasets seqeval evaluate accelerate --quiet
!pip install -q nltk --quiet

In [4]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
print(f"✅ PyTorch: {torch.__version__}")

✅ Device: cpu
✅ PyTorch: 2.11.0+cpu


In [5]:
# Task 1: Download CoNLL-2000 corpus from NLTK
nltk.download("conll2000")   # Chunking corpus: train.txt + test.txt
nltk.download("punkt")
from nltk.corpus import conll2000
# Peek at a sentence — each word is (token, pos_tag, chunk_tag)
sample_sent = conll2000.iob_sents()[0]
print("Sample sentence (word, POS, chunk):")
for word, pos, chunk in sample_sent[:8]:
    print(f"   {word:<15} {pos:<8} {chunk}")

Sample sentence (word, POS, chunk):
   Confidence      NN       B-NP
   in              IN       B-PP
   the             DT       B-NP
   pound           NN       I-NP
   is              VBZ      B-VP
   widely          RB       I-VP
   expected        VBN      I-VP
   to              TO       I-VP


[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\saimo\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\saimo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
# Build label lists from the full corpus
all_sents = conll2000.iob_sents()  # list of [(word, pos, chunk), ...]

# Collect unique labels (sorted for consistent id mapping)
all_pos_tags   = sorted(set(pos   for sent in all_sents for _, pos, _     in sent))
all_chunk_tags = sorted(set(chunk for sent in all_sents for _, _,   chunk in sent))

POS_LABEL_LIST   = all_pos_tags
CHUNK_LABEL_LIST = all_chunk_tags

pos_label2id   = {l: i for i, l in enumerate(POS_LABEL_LIST)}
chunk_label2id = {l: i for i, l in enumerate(CHUNK_LABEL_LIST)}

print(f"POS tag classes   ({len(POS_LABEL_LIST)}): {POS_LABEL_LIST}")
print(f"Chunk tag classes ({len(CHUNK_LABEL_LIST)}): {CHUNK_LABEL_LIST}")

POS tag classes   (44): ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']
Chunk tag classes (23): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


In [7]:
# Convert NLTK corpus to list of dicts
def nltk_to_records(sents, pos_l2id, chunk_l2id):
    records = []
    for sent in sents:
        tokens     = [w   for w, _, _ in sent]
        pos_ids    = [pos_l2id[p]   for _, p, _ in sent]
        chunk_ids  = [chunk_l2id[c] for _, _, c in sent]
        records.append({
            "tokens"    : tokens,
            "pos_tags"  : pos_ids,
            "chunk_tags": chunk_ids,
        })
    return records

# CoNLL-2000 only has train and test splits in NLTK
train_sents = conll2000.iob_sents("train.txt")
test_sents  = conll2000.iob_sents("test.txt")

all_train_records = nltk_to_records(train_sents, pos_label2id, chunk_label2id)
test_records      = nltk_to_records(test_sents,  pos_label2id, chunk_label2id)

# 90/10 split for train/validation
split_idx    = int(len(all_train_records) * 0.9)
train_records = all_train_records[:split_idx]
val_records   = all_train_records[split_idx:]

print(f"Train   : {len(train_records):,} sentences")
print(f"Val     : {len(val_records):,} sentences")
print(f"Test    : {len(test_records):,} sentences")

Train   : 8,042 sentences
Val     : 894 sentences
Test    : 2,012 sentences


In [8]:
# Build HuggingFace DatasetDict
raw_dataset = DatasetDict({
    "train"     : Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
    "test"      : Dataset.from_list(test_records),
})
print(raw_dataset)

# Verify a sample
ex = raw_dataset["train"][0]
print("\nFirst training sentence:")
print("  tokens    :", ex["tokens"][:8])
print("  pos_tags  :", ex["pos_tags"][:8],  "→", [POS_LABEL_LIST[i]   for i in ex["pos_tags"][:8]])
print("  chunk_tags:", ex["chunk_tags"][:8], "→", [CHUNK_LABEL_LIST[i] for i in ex["chunk_tags"][:8]])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 8042
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 894
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 2012
    })
})

First training sentence:
  tokens    : ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to']
  pos_tags  : [18, 13, 10, 18, 38, 26, 36, 31] → ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO']
  chunk_tags: [5, 6, 5, 16, 10, 21, 21, 21] → ['B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'I-VP', 'I-VP']


In [9]:
#Tokenizer & label alignment
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
# Choose task: "pos" or "chunk"
TASK         = "pos"           # ← change to "chunk" for chunking model
LABEL_LIST   = POS_LABEL_LIST   if TASK == "pos" else CHUNK_LABEL_LIST
LABEL_COLUMN = "pos_tags"       if TASK == "pos" else "chunk_tags"

print(f"Task: {TASK.upper()} Tagging  |  {len(LABEL_LIST)} classes")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,   # input is already split into words
        max_length=128,
        padding="max_length",
    )
    aligned_labels = []
    for i, label_ids in enumerate(examples[LABEL_COLUMN]):
        word_ids   = tokenized_inputs.word_ids(batch_index=i)
        prev_word  = None
        label_row  = []
        for word_idx in word_ids:
            if word_idx is None:              # special token
                label_row.append(-100)
            elif word_idx != prev_word:       # first subword → real label
                label_row.append(label_ids[word_idx])
            else:                             # continuation subword → ignore
                label_row.append(-100)
            prev_word = word_idx
        aligned_labels.append(label_row)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs


# Apply to all splits
tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)
print("\nTokenization complete!")
sample = tokenized_datasets["train"][0]
print("Fields         :", list(sample.keys()))
print("input_ids len  :", len(sample["input_ids"]))
print("attention_mask :", len(sample["attention_mask"]))
print("labels len     :", len(sample["labels"]))
# Confirm -100 on subwords / special tokens
print("Labels (first 15):", sample["labels"][:15])


Task: POS Tagging  |  44 classes


Map: 100%|████████████████████████████████████████████████████████████████| 2012/2012 [00:00<00:00, 2242.10 examples/s]


Tokenization complete!
Fields         : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids len  : 128
attention_mask : 128
labels len     : 128
Labels (first 15): [-100, 18, 13, 10, 18, 38, 26, 36, 31, 33, 10, 14, 18, 13, 18]


In [10]:
# Task 3: id-label mappings + model load
id2label = {i: l for i, l in enumerate(LABEL_LIST)}
label2id = {l: i for i, l in enumerate(LABEL_LIST)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

print(f"Model : {MODEL_CHECKPOINT}")
print(f"   Params: {model.num_parameters():,}")
print(f"   Head  : Linear({model.config.hidden_size} → {model.config.num_labels})")
print(f"   Labels: {list(id2label.values())}")

Model : distilbert-base-uncased
   Params: 66,396,716
   Head  : Linear(768 → 44)
   Labels: ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']


In [11]:
# Task 4: seqeval metric
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [LABEL_LIST[pred] for pred, lbl in zip(pred_row, lbl_row) if lbl != -100]
        for pred_row, lbl_row in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[lbl] for lbl in lbl_row if lbl != -100]
        for lbl_row in labels
    ]

    results = seqeval.compute(
        predictions=true_preds,
        references=true_labels,
        zero_division=0
    )
    return {
        "precision": results["overall_precision"],
        "recall"   : results["overall_recall"],
        "f1"       : results["overall_f1"],
        "accuracy" : results["overall_accuracy"],
    }

In [12]:
# Training arguments
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir=f"./distilbert-{TASK}-tagging",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,          # small LR to avoid destroying pretrained weights
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none",
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1000)),
    eval_dataset=tokenized_datasets["validation"].select(range(300)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"Starting {TASK.upper()} training...")
train_result = trainer.train()
print(f"\nTraining done! Loss: {train_result.training_loss:.4f}")

Starting POS training...
{'loss': '2.463', 'grad_norm': '2.693', 'learning_rate': '4.643e-06', 'epoch': '0.8'}
{'train_runtime': '442.6', 'train_samples_per_second': '2.259', 'train_steps_per_second': '0.282', 'train_loss': '2.257', 'epoch': '1'}

Training done! Loss: 2.2568


In [ ]:
# Task 5: Evaluate on test set
preds, labels, _ = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(preds, axis=2)

true_preds = [
    [LABEL_LIST[p] for p, l in zip(pr, lr) if l != -100]
    for pr, lr in zip(preds, labels)
]
true_labels = [
    [LABEL_LIST[l] for l in lr if l != -100]
    for lr in labels
]
results = seqeval.compute(predictions=true_preds, references=true_labels)
print(f"{'='*48}")
print(f"  {TASK.upper()} TAGGING — Test Set Evaluation")
print(f"{'='*48}")
print(f"  Precision : {results['overall_precision']:.4f}")
print(f"  Recall    : {results['overall_recall']:.4f}")
print(f"  F1 Score  : {results['overall_f1']:.4f}")
print(f"  Accuracy  : {results['overall_accuracy']:.4f}")
print(f"{'='*48}")
print("\nPer-class breakdown:")
for cls, m in results.items():
    if isinstance(m, dict):
        print(f"   {cls:<12} P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  (n={m['number']})")

In [ ]:
# Task 6: Inference pipeline
tag_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",  # merges ##subwords back to whole words
    device=0 if device == "cuda" else -1,
)
def predict_tags(sentence):
    out = tag_pipeline(sentence)
    print(f"\n'{sentence}'")
    print(f"{'Word':<22} {'Tag':<12} Score")
    print("-" * 44)
    for item in out:
        word = item["word"].lstrip("##")
        print(f"{word:<22} {item['entity_group']:<12} {item['score']:.3f}")

predict_tags("John works at Google in California")
predict_tags("The quick brown fox jumps over the lazy dog")
predict_tags("Apple released a new iPhone model last Tuesday")

In [ ]:
# Helper: train + evaluate one task, return metrics dict
def run_task(task_name):
    label_list = POS_LABEL_LIST if task_name == "pos" else CHUNK_LABEL_LIST
    label_col  = "pos_tags"     if task_name == "pos" else "chunk_tags"

    def _align(examples):
        tok = tokenizer(
            examples["tokens"], truncation=True, is_split_into_words=True,
            max_length=128, padding="max_length",
        )
        aligned = []
        for i, lbl_ids in enumerate(examples[label_col]):
            word_ids = tok.word_ids(batch_index=i)
            prev, row = None, []
            for wid in word_ids:
                if wid is None:       row.append(-100)
                elif wid != prev:     row.append(lbl_ids[wid])
                else:                 row.append(-100)
                prev = wid
            aligned.append(row)
        tok["labels"] = aligned
        return tok

    ds = raw_dataset.map(_align, batched=True,
                         remove_columns=raw_dataset["train"].column_names)

    _id2label = {i: l for i, l in enumerate(label_list)}
    _label2id = {l: i for i, l in enumerate(label_list)}
    _model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=len(label_list),
        id2label=_id2label, label2id=_label2id,
    ).to(device) # Move model to the detected device (GPU/CPU)

    def _metrics(p):
        pr, lb = p
        pr = np.argmax(pr, axis=2)
        tp = [[label_list[x] for x, y in zip(r, s) if y != -100] for r, s in zip(pr, lb)]
        tl = [[label_list[y] for y in s if y != -100] for s in lb]
        r  = seqeval.compute(predictions=tp, references=tl)
        return {"precision": r["overall_precision"], "recall": r["overall_recall"],
                "f1": r["overall_f1"], "accuracy": r["overall_accuracy"]}

    _args = TrainingArguments(
        output_dir=f"./distilbert-{task_name}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=2e-5, weight_decay=0.01, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True,
        logging_steps=200, report_to="none", push_to_hub=False,
    )
    _trainer = Trainer(
        model=_model, args=_args,
        train_dataset=ds["train"], eval_dataset=ds["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=_metrics,
    )
    _trainer.train()

    preds, lbls, _ = _trainer.predict(ds["test"])
    preds = np.argmax(preds, axis=2)
    tp = [[label_list[p] for p, l in zip(pr, lr) if l != -100] for pr, lr in zip(preds, lbls)]
    tl = [[label_list[l] for l in lr if l != -100] for lr in lbls]
    metrics = seqeval.compute(predictions=tp, references=tl)

    # Explicitly move model to CPU and clear GPU cache
    _model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    del _model # Delete model instance
    del _trainer # Delete trainer instance
    import gc
    gc.collect() # Force garbage collection

    return metrics


print("Training POS model...")
pos_res = run_task("pos")

print("\nTraining Chunking model...")
chunk_res = run_task("chunk")

In [ ]:
# Side-by-side comparison table
print(f"\n{'='*55}")
print(f"{'Metric':<15} {'POS Tagging':>18} {'Chunking':>18}")
print(f"{'='*55}")
for key in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
    label = key.replace("overall_", "").capitalize()
    print(f"{label:<15} {pos_res[key]:>18.4f} {chunk_res[key]:>18.4f}")
print(f"{'='*55}")